<a href="https://colab.research.google.com/github/peacewhile/PHM-Learning-Task/blob/main/NGAFID_DATASET_MINIROCKET_EXAMPLE7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/hyang0129/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)

!pip install tsai -q

In [ ]:
!pip install --upgrade ipython -q

In [ ]:
import sys
sys.path.append('/content/NGAFIDDATASET')

In [ ]:
%load_ext autoreload

In [ ]:
%autoreload
from tsai.basics import *
from tsai.models.MINIROCKET_Pytorch import *
from tsai.models.utils import *
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!tar -xzf /content/drive/MyDrive/NGAFIDDATASET/2days.tar.gz -C /content/

In [ ]:
import sys
sys.path.append('/content/NGAFIDDATASET')

import numpy as np
import pandas as pd
import pickle

# 手动构建一个简单的 dm 对象
class SimpleDM:
    pass

dm = SimpleDM()

# 注入元数据
dm.flight_header_df = pd.read_csv(
    '/content/2days/flight_header.csv',
    index_col='Master Index'
)

# 注入时序数据
with open('/content/2days/flight_data.pkl', 'rb') as f:
    dm.data_dict = pickle.load(f)

# 加载归一化参数
stats = pd.read_csv('/content/2days/stats.csv', index_col=0)
print(stats.head())
print(stats.shape)

In [ ]:
import numpy as np
stats_numeric = stats.drop(columns=['cluster'])
dm.mins = stats_numeric.loc[1].values.astype(np.float32)
dm.maxs = stats_numeric.loc[0].values.astype(np.float32)

In [ ]:
import types

def get_numpy_dataset(self, fold, training):
    if training:
        mask = self.flight_header_df['fold'] != fold
    else:
        mask = self.flight_header_df['fold'] == fold
    subset = self.flight_header_df[mask]
    data = [self.data_dict[idx] for idx in subset.index]
    return {
        'data':         data,
        'target_class': subset['target_class'].values,
        'before_after': subset['before_after'].values,
        'fold':         subset['fold'].values
    }

dm.get_numpy_dataset = types.MethodType(get_numpy_dataset, dm)

In [ ]:
from fastai.callback.progress import CSVLogger

from tqdm.autonotebook import tqdm

In [ ]:
def pad_or_truncate(data_list, target_len=3000):
    result = []
    for arr in data_list:
        T = arr.shape[0]
        if T >= target_len:
            arr = arr[:target_len, :]
        else:
            pad = np.zeros((target_len - T, arr.shape[1]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        result.append(arr)
    return np.array(result, dtype=np.float32).transpose(0, 2, 1)

In [ ]:
import gc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

save_path = '.'
model_name = 'MINIROCKET_BINARY'

mins23 = dm.mins[:23].reshape(-1, 1)
maxs23 = dm.maxs[:23].reshape(-1, 1)

# 收集每折指标
fold_accuracies = []
fold_f1s = []
fold_aucs = []

# 收集最后一折预测结果（用于混淆矩阵）
last_fold_pred_labels = None
last_fold_test_Y = None

for fold in tqdm(range(5)):

    save_filename = save_path + '%s_%i' % (model_name, fold)

    train_dict = dm.get_numpy_dataset(fold=fold, training=True)
    test_dict  = dm.get_numpy_dataset(fold=fold, training=False)

    train_X = pad_or_truncate(train_dict['data'], target_len=200)
    train_X = (train_X - mins23) / (maxs23 - mins23)
    train_X = np.nan_to_num(train_X, copy=False)

    test_X = pad_or_truncate(test_dict['data'], target_len=200)
    test_X = (test_X - mins23) / (maxs23 - mins23)
    test_X = np.nan_to_num(test_X, copy=False)

    train_Y = np.array(train_dict['before_after'])
    test_Y  = np.array(test_dict['before_after'])

    splits = [list(np.arange(len(train_Y)))]
    splits.append(list(np.arange(len(test_Y)) + len(train_Y)))

    torch.cuda.empty_cache()
    mrf = MiniRocketFeatures(train_X.shape[1], train_X.shape[2]).to(default_device())
    chunksize = 32

    mrf.fit(train_X, chunksize=chunksize)

    X_feat = get_minirocket_features(np.concatenate([train_X, test_X]), mrf, chunksize=chunksize, to_np=True)

    Y = np.concatenate([train_Y, test_Y])

    PATH = Path("./models/MRF.pt")
    PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mrf.state_dict(), PATH)

    tfms = [None, TSClassification()]
    batch_tfms = TSStandardize(by_sample=True)
    dls = get_ts_dls(X_feat, Y, splits=splits, tfms=tfms, batch_tfms=batch_tfms)
    model = build_ts_model(MiniRocketHead, dls=dls)

    learn = Learner(dls, model, metrics=accuracy, cbs=[ShowGraph(), CSVLogger(save_filename)])

    results = learn.fit_one_cycle(200, 2.5e-5)

    # 获取测试集预测结果
    test_dl = dls.test_dl(X_feat[splits[1]])
    preds, _ = learn.get_preds(dl=test_dl)
    pred_probs  = preds[:, 1].numpy()
    pred_labels = preds.argmax(dim=1).numpy()

    # 计算三个指标
    acc = (pred_labels == test_Y).mean()
    f1  = f1_score(test_Y, pred_labels, average='binary')
    auc = roc_auc_score(test_Y, pred_probs)

    fold_accuracies.append(acc)
    fold_f1s.append(f1)
    fold_aucs.append(auc)

    print(f"Fold {fold} → Accuracy: {acc:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}")

    # 保存最后一折预测结果
    if fold == 4:
        last_fold_pred_labels = pred_labels
        last_fold_test_Y      = test_Y

    # 每折结束后释放内存
    del train_X, test_X, X_feat, mrf, model, learn, dls, results
    torch.cuda.empty_cache()
    gc.collect()

# ========== 汇总报告 ==========
fold_accuracies = np.array(fold_accuracies)
fold_f1s        = np.array(fold_f1s)
fold_aucs       = np.array(fold_aucs)

print("\n========== 五折交叉验证结果 ==========")
print(f"{'Fold':<8} {'Accuracy':<12} {'F1':<12} {'ROC-AUC':<12}")
print("-" * 44)
for i in range(5):
    print(f"Fold {i:<4} {fold_accuracies[i]:<12.4f} {fold_f1s[i]:<12.4f} {fold_aucs[i]:<12.4f}")
print("-" * 44)
print(f"{'均值':<8} {fold_accuracies.mean():<12.4f} {fold_f1s.mean():<12.4f} {fold_aucs.mean():<12.4f}")
print(f"{'标准差':<8} {fold_accuracies.std():<12.4f}  {fold_f1s.std():<12.4f}  {fold_aucs.std():<12.4f}")
print("======================================")

# ========== 图1：五折指标柱状图 ==========
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
folds = [f'Fold {i}' for i in range(5)]
metrics = {
    'Accuracy': fold_accuracies,
    'F1':       fold_f1s,
    'ROC-AUC':  fold_aucs
}

for ax, (name, values) in zip(axes, metrics.items()):
    bars = ax.bar(folds, values,
                  color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'])
    ax.axhline(y=values.mean(), color='red', linestyle='--',
               label=f'均值={values.mean():.4f}')
    ax.set_title(f'{name} 各折结果')
    ax.set_ylabel(name)
    ax.set_ylim(0, 1)
    ax.legend()
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('五折交叉验证结果', fontsize=14)
plt.tight_layout()
plt.savefig('fold_metrics.png', dpi=150)
plt.show()

# ========== 图2：混淆矩阵（最后一折）==========
cm = confusion_matrix(last_fold_test_Y, last_fold_pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['维护后(0)', '维护前(1)'],
            yticklabels=['维护后(0)', '维护前(1)'])
plt.title('混淆矩阵（Fold 4）')
plt.ylabel('真实标签')
plt.xlabel('预测标签')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

# ========== 图3：均值±标准差汇总图 ==========
fig, ax = plt.subplots(figsize=(8, 5))
metric_names = ['Accuracy', 'F1', 'ROC-AUC']
means = [fold_accuracies.mean(), fold_f1s.mean(), fold_aucs.mean()]
stds  = [fold_accuracies.std(),  fold_f1s.std(),  fold_aucs.std()]

bars = ax.bar(metric_names, means, yerr=stds, capsize=10,
              color=['#4C72B0', '#DD8452', '#55A868'],
              error_kw={'elinewidth': 2, 'ecolor': 'black'})
ax.set_ylim(0, 1)
ax.set_title('五折交叉验证 均值 ± 标准差')
ax.set_ylabel('分数')

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + std + 0.02,
            f'{mean:.4f}±{std:.4f}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('metrics_summary.png', dpi=150)
plt.show()
